# patient-zero-to-insight

A walk through global COVID-19 case and death data — from raw CSVs to three specific, defensible findings. Data: Johns Hopkins CSSE, 22 Jan 2020 – 9 Mar 2023.

## 1. Load & inspect

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

plt.style.use('seaborn-v0_8-whitegrid')

confirmed = pd.read_csv('../data/confirmed_global.csv')
deaths = pd.read_csv('../data/deaths_global.csv')

confirmed.head()

## 2. Clean

Collapse province-level rows into country totals, and convert the date columns into a proper datetime-indexed time series.

In [ ]:
date_cols = confirmed.columns[4:]

def to_country_timeseries(df):
    g = df.groupby('Country/Region')[date_cols].sum()
    g.columns = pd.to_datetime(g.columns, format='%m/%d/%y')
    return g

conf_by_country = to_country_timeseries(confirmed)
deaths_by_country = to_country_timeseries(deaths)

conf_by_country.iloc[:5, -5:]

## 3. Explore

Daily new cases via `.diff()`, smoothed with a 7-day rolling average to cut day-of-week reporting noise.

In [ ]:
global_daily_confirmed = conf_by_country.sum(axis=0)
global_daily_new = global_daily_confirmed.diff().fillna(0)
global_daily_new_smooth = global_daily_new.rolling(7).mean()

peak_date = global_daily_new_smooth.idxmax()
peak_value = global_daily_new_smooth.max()
print('Global peak (7-day avg):', peak_date.strftime('%d %b %Y'), f'{peak_value:,.0f} cases/day')

In [ ]:
totals = conf_by_country.iloc[:, -1].sort_values(ascending=False)
top5 = totals.head(5)

death_totals = deaths_by_country.iloc[:, -1]
cfr = (death_totals / conf_by_country.iloc[:, -1] * 100)
top5_cfr = cfr.loc[top5.index]

pd.DataFrame({'total_confirmed': top5, 'cfr_percent': top5_cfr.round(2)})

## 4. Visualize

In [ ]:
fig, ax = plt.subplots(figsize=(10,5))
ax.plot(global_daily_new_smooth.index, global_daily_new_smooth.values, color='#1a1a1a', linewidth=1.8)
ax.fill_between(global_daily_new_smooth.index, global_daily_new_smooth.values, color='#1a1a1a', alpha=0.08)
ax.set_title('global daily new confirmed cases (7-day avg)', fontsize=13, loc='left', fontweight='bold')
ax.set_ylabel('new cases / day')
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=4))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('../assets/global_trend.png', dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10,5))
colors = ['#1a1a1a', '#555555', '#8a8a8a', '#b0b0b0', '#d0d0d0']
for i, c in enumerate(top5.index):
    ax.plot(conf_by_country.columns, conf_by_country.loc[c], label=c, color=colors[i], linewidth=1.8)
ax.set_title('cumulative confirmed cases — top 5 countries', fontsize=13, loc='left', fontweight='bold')
ax.set_ylabel('total confirmed')
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=4))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.legend(frameon=False, fontsize=9)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('../assets/top5_countries.png', dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
vals = top5_cfr.sort_values(ascending=True)
ax.barh(vals.index, vals.values, color='#1a1a1a')
ax.set_title('case fatality rate — top 5 by case volume', fontsize=13, loc='left', fontweight='bold')
ax.set_xlabel('deaths / confirmed cases (%)')
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('../assets/cfr_comparison.png', dpi=150)
plt.show()

## 5. Interpret

**1. The Omicron wave was the sharpest spike in the dataset.** Global daily new cases (7-day avg) peaked at ~3.44M/day on 24 Jan 2022 — over 4x the previous largest wave (Delta, ~830K/day, mid-2021).

**2. Case totals don't track population the way you'd expect.** The US recorded 103.8M total confirmed cases — more than double India's 44.7M, despite India's population being ~4x larger. Points to testing/reporting capacity as much as actual spread.

**3. Case fatality rate varied far more than case volume.** Among the top 5 countries by cases, Brazil's CFR (1.89%) was over 4x France's (0.42%) and Germany's (0.44%) — likely reflects healthcare capacity and reporting standards more than the virus itself.